# 1. TEMPORAL 2×2 FACTORIAL ABLATION — AeroSync-Mamba v22

v22 starts from v18. Four lag-bias variants have now failed to show a benefit. This notebook asks **where useful
temporal information, if any, enters the model**. It crosses two independent switches:

| Condition | HTT time embedding φ(τ) | LACA lag bias β·LN(g(Δτ)) | |
|---|---|---|---|
| **A** | ON | OFF | control = v18 at β = 0 |
| **B** | OFF | ON | |
| **C** | ON | ON | |
| **D** | OFF | OFF | no timestamps reach the model |

Three paired questions, each asked of macro-F1 and of Class 4 recall:
- **Q1. Does removing HTT's time embedding hurt?** A vs D, with LACA off in both.
- **Q2. Does LACA's lag bias add anything beyond HTT?** C vs A.
- **Q3. Can LACA's lag bias work without HTT?** B vs D.

**Where the switches act.** From v18's code:
- **HTT:** `HeterogeneousTemporalTokenization.forward` returns `norm(z + e_m + time_embedding(tau))`, where
  `time_embedding` is `ContinuousTimeEmbedding`. With `HTT_TIME_EMBED_ENABLED = False` it returns `norm(z + e_m)`.
  Nothing else in HTT changes: same modality embedding, same LayerNorm.
- **LACA:** `PairwiseLACA.forward` computes `softmax(scores + beta * lag_bias)`, where `lag_bias = lag_bias_term(tau_m,
  tau_n)` = LN_grid(g(Δτ)). With `LACA_LAG_BIAS_ENABLED = True` the scale is **β = 1.0**. With `False` it is exactly
  **0.0**, which is the same code path as v18's β = 0 runs, so Condition A stays bit-compatible with them.
- **No extra random draws.** Both modules are **built in every condition**, so every condition of a seed has the same
  initial weights and shuffle/dropout stream (checked). A disabled module just gets no use (HTT) or a 0 multiplier
  (LACA).
- **Every run checks** that the disabled contribution is exactly 0, both before training and after.

**Condition A: reused, verified first.** v18 had no HTT switch; its HTT always added φ(τ). So v18's β = 0.0
configuration is exactly Condition A. The v18 CSV provides the 4 metrics, and v20 provides the confusion matrices,
since it regenerated this exact control and matched all 4 metrics. To remove any doubt, **Condition A is retrained for
seed 42 only**, which also supplies the H2 model:
- If it reproduces v18 (±0.0005 on every metric), seeds 123, 2024, 7 and 99 reuse the v18 rows.
- If not, Condition A is **retrained for every seed**.

That is **16 training runs**, or **20** if the check fails. Runs are grouped by seed (A, B, C, D per seed).

**H2 robustness.** Seed 42 of each condition gets the existing temporal-shift sweep from v4/v5: `tau_s ← clamp(tau_s +
δ, 0, 1)` for δ ∈ {0, 0.05, 0.10, 0.20, 0.30}, with macro-F1 retention relative to δ = 0. Condition D uses no
timestamps at all, so its curve **must** be exactly flat. That serves as one more check that both switches are off.

**Class 4 recall** is recorded only as a hard-case diagnostic. Nothing in this notebook optimizes toward it.

| Decision | Value | Evidence status |
|---|---|---|
| Timestamps into HTT / LACA | Real per-window audio and IMU timestamps (not a synthetic `linspace`). **v22: HTT's time embedding φ(τ) is an ON/OFF factor.** | CORRECTNESS FIX (earlier versions passed an identical synthetic grid for every window) |
| Fusion backbone | Real `mamba_ssm.Mamba` selective-scan SSM; pure-PyTorch fallback only when no CUDA GPU is available (printed unmissably) | CORRECTNESS FIX (the earlier gated-conv block was not an SSM) |
| Audio bin selection | Strongest-N FFT bins by mean magnitude, computed once from **training** audio only, then frozen and reused for val/test | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): strongest-64 0.8736 vs first-64 0.8494, +0.0242 (bin-selection study) |
| Frequency budget N | 64 bins | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v6: N=64 0.8966 vs N=48 0.8863, **+0.0103**, which is below 0.0160. The other budgets clear it (N=32 +0.0249, N=128 +0.0166) except N=96 (+0.0142). |
| Audio compression | `sqrt(|rFFT|)` | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 sqrt 0.8966 vs log_db 0.8696 (+0.0270) and log1p 0.8649 (+0.0317) |
| Audio windowing | None; Hann windowing rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 no window 0.8649 vs Hann 0.8290 (+0.0359) |
| Audio normalization | None; z-score normalization rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 none 0.8966 vs z-score 0.8684 (+0.0282) |
| Audio time positions Ta | 64 per 2.56 s window | PROVISIONAL: v7 Ta=128 0.8967 vs Ta=64 0.8966 (−0.0001, a tie). The cheaper Ta=64 is kept. |
| Sensor input | Raw 6 channels (accel x/y/z, gyro x/y/z), Ts=128; derived magnitude features rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v8 raw6 0.8966 vs best derived set 0.8740 (+0.0226) |
| LACA lag bias β·LN(g(Δτ)) | **v22: ON/OFF factor** (ON = β 1.0 fixed from epoch 1, no warm-up, not learnable) | NOT ESTABLISHED as a global term (v17/v18/v19) or as query-dependent lags (v21). **Crossed with the HTT time embedding in this notebook.** |
| LACA lag-bias form g(Δτ) | mlp `Linear(1,32) → Tanh → Linear(32,1)`, 97 params. **v18:** final Linear zero-initialized, output LayerNorm over each window's (T_m, T_n) grid (0 params) | PROVISIONAL: v10 mlp vs rbf +0.0029, vs bucketed +0.0030 (below 0.0160). The v18 stabilization is under test. |
| LACA heads | 4 | PROVISIONAL: v11 h=1 was +0.0008 (one test window) and h=4 was kept. h=2/8/16 were 0.006–0.012 worse (below 0.0160). |
| Mamba fusion | 2 blocks, d_state=16, d_conv=2 (expand 2) | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v12: 2 vs 4 blocks +0.0023, d_state 16 vs 8 +0.0172, d_conv 2 vs 4 +0.0059. |
| λ_align (InfoNCE) | 0.1 | PROVISIONAL: v13 removing L_align cost −0.0088, which is below 0.0160. λ=0.05 was −0.0271 and λ=0.2 was −0.0070. |
| λ_cons (KL consistency) | 0.1 | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v13 grid: λ=0.2 +0.0008, λ=0.05 −0.0232. Removing L_cons cost −0.0165. |
| Ordinal-severity loss | Not used (rejected) | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v14 best λ_ord>0 0.8859 vs none 0.9048 (−0.0189); adjacent-class errors rose 94 → 113 |
| Classification loss | Cross-entropy weighted by inverse **training-set** class frequency | CORRECTNESS FIX (class-imbalance handling) |
| Seeding | Python `random`, numpy, torch CPU + all CUDA devices, cuDNN deterministic; reset per seed at run start and right before model construction. The mission split has its own fixed seed (42), so every seed sees the same train/val/test windows. | CORRECTNESS FIX (reproducibility) |

## 2. Configuration constants & environment
Every fixed value lives in the first cell below. The second installs `mamba_ssm` and reports which fusion backbone is active.

In [1]:
import copy
import csv
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
import urllib.request
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==============================================================================
# FIXED CONFIGURATION — every finalized choice (see the summary table above)
# ==============================================================================
# --- 2x2 factorial: HTT time embedding x LACA lag bias, 5 seeds, grouped by seed ---
SEEDS = [42, 123, 2024, 7, 99]           # same 5 seeds as every prior multi-seed experiment
CONDITIONS = {                           # name: (HTT_TIME_EMBED_ENABLED, LACA_LAG_BIAS_ENABLED)
    "A": (True, False),                  # control = v18 at beta = 0
    "B": (False, True),
    "C": (True, True),
    "D": (False, False),
}
RUN_ORDER = [(seed, cond) for seed in SEEDS for cond in CONDITIONS]   # A, B, C, D per seed
REUSE_CHECK_SEED = 42                    # Condition A is retrained for this seed to verify the v18 reuse
H2_SEED = 42                             # the H2 temporal-shift sweep runs on this seed's model of every condition
H2_OFFSETS = [0.0, 0.05, 0.10, 0.20, 0.30]   # tau_s <- clamp(tau_s + delta, 0, 1), as in v4/v5 (H2)
SPLIT_SEED = 42                          # mission-level train/val/test split (independent of the training seed)

# --- Dataset / split ---
DATASET_REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"
WORK_DIR = Path("/kaggle/working")
DATASET_BASE = WORK_DIR / "tii_uav_dataset"
DATASET_ROOT = DATASET_BASE / "Dataset"
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
N_CLASSES = 5                            # propeller-fault severity 0..4

# --- Windowing ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
MIN_SENSOR_SAMPLES_PER_WINDOW = 10

# --- Audio: strongest-64 bins (train-only), sqrt(|rFFT|), no window, no normalization ---
N_AUDIO_BINS = 64                        # frequency axis
TA_AUDIO = 64                            # audio time positions per window

# --- Sensor: raw 6 channels (accel x/y/z, gyro x/y/z), no derived features ---
N_SENSOR_CHANNELS = 6
TS_SENSOR = 128                          # sensor time positions per window
SENSOR_STD_EPS = 1e-6

# --- Model ---
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 1.0                          # beta when LACA_LAG_BIAS_ENABLED (fixed from epoch 1: no warm-up, not learnable)
LAG_BIAS_HIDDEN = 32                     # mlp g(Δτ): Linear(1,32) -> Tanh -> Linear(32,1) = 97 params
LAG_BIAS_PARAMS = 97                     # the LayerNorm added in v18 has no parameters -> still 97
LAG_BIAS_LN_EPS = 1e-5                   # LayerNorm over each window's (T_m, T_n) lag-bias grid, no affine
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496          # 2 x 437,248 (real mamba_ssm)
DROPOUT = 0.15

# --- Loss: L_cls (class-weighted CE) + 0.1 * L_align (InfoNCE) + 0.1 * L_cons (KL) ---
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07

# --- Optimization ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- Outputs ---
CHECKPOINT_DIR = WORK_DIR / "v22_checkpoints"
PROBS_DIR = WORK_DIR / "v22_test_probs"
RESULTS_CSV = WORK_DIR / "temporal_2x2_ablation_results.csv"
H2_CSV = WORK_DIR / "temporal_2x2_h2_robustness.csv"
for _d in (CHECKPOINT_DIR, PROBS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Condition A candidates for reuse: v18's beta=0.0 runs (HTT time embedding always ON in v18) ---
# 4 metrics + init / RNG fingerprints: Artifacts/Version 18 CSV. Confusion matrices: Artifacts/Version 20 CSV (the same
# runs, regenerated in v20 with all 4 metrics identical to v18).
V18_BETA0_RUNS = {
    42: {"accuracy": 0.9088495575221239, "macro_f1": 0.9057330017972403, "balanced_accuracy": 0.9030033658152373, "macro_auroc": 0.9891867537737307,
          "confusion_matrix": [[283, 0, 1, 0, 2], [3, 197, 2, 1, 2], [0, 1, 172, 35, 2], [0, 0, 8, 204, 16], [0, 0, 2, 28, 171]],
          "init_sha1": "5850841e699f", "rng_sha1": "5641b401720d"},
    123: {"accuracy": 0.8876106194690265, "macro_f1": 0.8826825575079713, "balanced_accuracy": 0.8790467961967972, "macro_auroc": 0.9861866752615566,
          "confusion_matrix": [[283, 1, 1, 0, 1], [2, 196, 6, 1, 0], [0, 2, 177, 30, 1], [0, 0, 8, 203, 17], [0, 1, 2, 54, 144]],
          "init_sha1": "74be0883c62b", "rng_sha1": "0a4b3be44b18"},
    2024: {"accuracy": 0.8805309734513275, "macro_f1": 0.8763643000413779, "balanced_accuracy": 0.8715485268344848, "macro_auroc": 0.9866958753709998,
          "confusion_matrix": [[282, 1, 0, 0, 3], [2, 189, 12, 1, 1], [0, 0, 168, 39, 3], [0, 0, 6, 206, 16], [0, 0, 0, 51, 150]],
          "init_sha1": "92852955eef1", "rng_sha1": "619c0e0a6b4d"},
    7: {"accuracy": 0.9061946902654867, "macro_f1": 0.9019350016739314, "balanced_accuracy": 0.9008047135870789, "macro_auroc": 0.9897445660698647,
          "confusion_matrix": [[284, 1, 0, 0, 1], [2, 196, 7, 0, 0], [0, 3, 183, 22, 2], [0, 0, 16, 191, 21], [0, 0, 3, 28, 170]],
          "init_sha1": "7f5d0cd2e7cb", "rng_sha1": "9fdcf777d285"},
    99: {"accuracy": 0.8690265486725663, "macro_f1": 0.8638026478536822, "balanced_accuracy": 0.8598248510633313, "macro_auroc": 0.9829841151452052,
          "confusion_matrix": [[282, 0, 1, 0, 3], [5, 190, 9, 0, 1], [0, 2, 164, 39, 5], [0, 0, 12, 197, 19], [0, 1, 3, 48, 149]],
          "init_sha1": "e40c78169f57", "rng_sha1": "81dcd4638a88"},
}
SAME_SEED_TOLERANCE = 0.0005             # same-seed reproducibility (code correctness), not a finding threshold
SEED_TO_SEED_STD_F1 = 0.0160             # v15/v16 measured seed-to-seed std of test macro-F1
PAIRED_ALPHA = 0.05
REFERENCE_SEED42_RNG_PREFIX = "5641b401720d"   # torch CPU RNG right after seed-42 model construction (v10-v15)
REFERENCE_DATA = {
    "bin_indices": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                    258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                    509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                    761, 762, 763, 764, 765, 766],
    "sensor_fingerprint": {
        "train": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
        "val":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
        "test":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
    },
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


def set_all_seeds(seed: int) -> None:
    """Seed python `random`, numpy, torch CPU and every CUDA device; deterministic cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print(f"TEMPORAL 2x2 ABLATION: conditions " + ", ".join(f"{c} (HTT-time {'ON' if h else 'OFF'}, LACA-lag "
      f"{'ON' if l else 'OFF'})" for c, (h, l) in CONDITIONS.items()) + f" x seeds {SEEDS} | results -> {RESULTS_CSV}")
print(f"LACA lag bias when ON: beta {LACA_BETA} fixed * LN_grid(g(Δτ)) (v18 form, zero-init g) | H2 sweep on seed {H2_SEED}: "
      f"offsets {H2_OFFSETS} -> {H2_CSV}")
print(f"Fixed configuration: audio strongest-{N_AUDIO_BINS}, Ta={TA_AUDIO}, sqrt, no window, no normalization "
      f"| sensor raw {N_SENSOR_CHANNELS} ch, Ts={TS_SENSOR} | LACA lag bias ON/OFF (beta {LACA_BETA}), {LACA_NUM_HEADS} heads, mlp g(Δτ) "
      f"{LAG_BIAS_PARAMS} params | Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) | "
      f"loss CE + {LAMBDA_ALIGN} InfoNCE + {LAMBDA_CONS} KL | {EPOCHS} epochs, batch {BATCH_SIZE}, AdamW {LEARNING_RATE}")

Device: cuda (Tesla T4)
TEMPORAL 2x2 ABLATION: conditions A (HTT-time ON, LACA-lag OFF), B (HTT-time OFF, LACA-lag ON), C (HTT-time ON, LACA-lag ON), D (HTT-time OFF, LACA-lag OFF) x seeds [42, 123, 2024, 7, 99] | results -> /kaggle/working/temporal_2x2_ablation_results.csv
LACA lag bias when ON: beta 1.0 fixed * LN_grid(g(Δτ)) (v18 form, zero-init g) | H2 sweep on seed 42: offsets [0.0, 0.05, 0.1, 0.2, 0.3] -> /kaggle/working/temporal_2x2_h2_robustness.csv
Fixed configuration: audio strongest-64, Ta=64, sqrt, no window, no normalization | sensor raw 6 ch, Ts=128 | LACA lag bias ON/OFF (beta 1.0), 4 heads, mlp g(Δτ) 97 params | Mamba 2 x (d_state 16, d_conv 2) | loss CE + 0.1 InfoNCE + 0.1 KL | 20 epochs, batch 32, AdamW 0.0001


In [2]:
# ==============================================================================
# mamba_ssm installation + fusion-backbone resolution.
# Order: an already-installed package, else a prebuilt wheel matched to this
# Python / torch / CUDA / C++ ABI (the reliable path on Kaggle), else a
# time-capped source build. If none succeeds, or there is no CUDA GPU, the
# pure-PyTorch fallback block is used and announced below.
# ==============================================================================
PIP_TIMEOUT_S = 240


def _env_tags():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    try:
        cxx11_abi = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except AttributeError:
        cxx11_abi = False
    return py_tag, torch_mm, torch.version.cuda, cxx11_abi


def _prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, cxx11_abi):
    """Best-matching .whl asset in `repo`'s GitHub releases (exact CUDA version first, then CUDA major)."""
    req = urllib.request.Request(f"https://api.github.com/repos/{repo}/releases",
                                 headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            releases = json.load(resp)
    except Exception as exc:
        print(f"    could not query github.com/{repo}/releases: {exc!r}")
        return None
    abi = f"cxx11abi{'TRUE' if cxx11_abi else 'FALSE'}"
    cuda_exact = f"cu{cuda_ver.replace('.', '')}"
    cuda_major = f"cu{cuda_ver.split('.')[0]}"
    exact, major_only = [], []
    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not (name.endswith(".whl") and py_tag in name and abi in name and f"torch{torch_mm}" in name):
                continue
            if cuda_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major in name:
                major_only.append(asset["browser_download_url"])
    chosen = exact or major_only
    return chosen[0] if chosen else None


def _pip_install(args):
    """pip with a hard timeout and captured output, so a stuck build can never hang the kernel."""
    try:
        proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                              timeout=PIP_TIMEOUT_S, capture_output=True, text=True, check=False)
    except subprocess.TimeoutExpired:
        print(f"    pip install {args} exceeded {PIP_TIMEOUT_S}s — killed")
        return False
    if proc.returncode != 0:
        print(f"    pip install {args} failed (exit {proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")
    return proc.returncode == 0


def install_mamba_ssm():
    if importlib.util.find_spec("mamba_ssm") is not None:
        return "already installed"
    tags = _env_tags()
    print(f"Installing mamba_ssm + causal-conv1d (python={tags[0]}, torch={tags[1]}, cuda={tags[2]}, cxx11abi={tags[3]})")
    urls = [_prebuilt_wheel_url(repo, *tags) for repo in ("Dao-AILab/causal-conv1d", "state-spaces/mamba")]
    if all(urls) and all(_pip_install([u, "--no-deps"]) for u in urls):
        return "prebuilt wheel"
    print("  no complete prebuilt-wheel match — trying a time-capped source build")
    if _pip_install(["causal-conv1d>=1.2.0", "--no-build-isolation"]) and _pip_install(["mamba-ssm", "--no-build-isolation"]):
        return "source build"
    return "failed"


MAMBA_CLASS = None
if torch.cuda.is_available():
    print(f"mamba_ssm install: {install_mamba_ssm()}")
    try:
        from mamba_ssm import Mamba as MAMBA_CLASS
    except ImportError as exc:
        print(f"mamba_ssm import failed: {exc!r}")

if MAMBA_CLASS is not None:
    FUSION_BACKBONE = "mamba_ssm.Mamba (real selective-scan SSM, CUDA)"
    print(f"✅ FUSION BACKBONE: {FUSION_BACKBONE}")
else:
    FUSION_BACKBONE = "FALLBACK pure-PyTorch gated-conv block (NOT a state-space model)"
    print("🚨" * 45)
    print(f"🚨  FUSION BACKBONE: {FUSION_BACKBONE}")
    print("🚨  " + ("no CUDA GPU detected" if not torch.cuda.is_available() else "mamba_ssm could not be installed/imported")
          + " — results will NOT match the reference numbers, which were produced with real mamba_ssm.")
    print("🚨" * 45)

Installing mamba_ssm + causal-conv1d (python=cp312, torch=2.10, cuda=12.8, cxx11abi=True)
    pip install ['https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl', '--no-deps'] failed (exit 1): ERROR: causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl is not a supported wheel on this platform.

  no complete prebuilt-wheel match — trying a time-capped source build
mamba_ssm install: source build
✅ FUSION BACKBONE: mamba_ssm.Mamba (real selective-scan SSM, CUDA)


## 3. Data loading & preprocessing
Mission-level split, strongest-64 bin selection from training audio only, 2.56 s windows with real timestamps, and sensor standardization with training-split statistics. Leakage and timestamp checks are included.

In [3]:
# ==============================================================================
# DATA — TII UAV Realistic Fault Dataset: parse missions -> mission-level split
# -> strongest-64 bins from TRAINING audio only (frozen) -> 2.56 s windows with
# real per-window timestamps -> datasets (sensor standardized with TRAINING stats).
# ==============================================================================
if not DATASET_ROOT.exists():
    print(f"Cloning {DATASET_REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO_URL, str(DATASET_BASE)], check=True)


def parse_mission(mission_dir: Path):
    """Real 6-axis IMU (seconds from mission start) + raw audio chunks with per-chunk timestamps.
    Audio chunk duration = the mission's IMU-measured duration / number of chunks (both come from the same
    continuous recording; the audio log carries no per-chunk timestamp)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro = [], [], []
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks = []
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    chunk_duration_s = (sensor_times[-1] - sensor_times[0]) / len(audio_chunks)
    audio_times = np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks}


def split_missions(dataset_root: Path):
    """Mission-level stratified split (70 / 15 / 15 per class) with its own fixed seed."""
    class_missions = defaultdict(list)
    for class_dir in sorted(dataset_root.iterdir()):
        if not class_dir.is_dir() or class_dir.name.startswith("."):
            continue
        try:
            label = int(class_dir.name)
        except ValueError:
            continue
        for mission_dir in sorted(class_dir.iterdir()):
            if mission_dir.is_dir() and not mission_dir.name.startswith("."):
                class_missions[label].append(mission_dir)

    random.seed(SPLIT_SEED)
    splits = {"train": {}, "val": {}, "test": {}}
    for label in sorted(class_missions):
        missions = class_missions[label]
        random.shuffle(missions)
        n_train = max(1, int(len(missions) * TRAIN_FRACTION))
        n_val = max(1, int(len(missions) * VAL_FRACTION))
        splits["train"][label] = missions[:n_train]
        splits["val"][label] = missions[n_train:n_train + n_val]
        splits["test"][label] = missions[n_train + n_val:]
        print(f"Class {label} ({len(missions)} flights) -> train {n_train}, val {len(splits['val'][label])}, "
              f"test {len(splits['test'][label])}")
    return splits


def select_strongest_bins(train_missions, n_bins):
    """Indices of the n_bins rFFT magnitude bins with the highest mean magnitude over TRAINING audio only
    (unsupervised energy, no labels). Returned sorted; frozen and reused for val/test."""
    magnitude_sum, n_chunks, fft_len = None, 0, None
    for mission in train_missions:
        mag = np.abs(np.fft.rfft(mission["audio_chunks"], axis=1))
        fft_len = mag.shape[1] if fft_len is None else min(fft_len, mag.shape[1])
        mag = mag[:, :fft_len]
        if magnitude_sum is not None and len(magnitude_sum) > fft_len:
            magnitude_sum = magnitude_sum[:fft_len]
        magnitude_sum = mag.sum(axis=0) if magnitude_sum is None else magnitude_sum + mag.sum(axis=0)
        n_chunks += mag.shape[0]
    mean_magnitude = magnitude_sum / n_chunks
    return np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)


def audio_features(audio_chunks, bin_indices):
    """raw chunk -> rFFT (no window) -> |.| -> frozen strongest bins -> sqrt (no normalization)."""
    mag = np.abs(np.fft.rfft(audio_chunks, axis=1))
    valid = bin_indices[bin_indices < mag.shape[1]]
    feats = np.zeros((mag.shape[0], len(bin_indices)), dtype=np.float32)
    feats[:, :len(valid)] = np.sqrt(mag[:, valid]).astype(np.float32)
    return feats


def slice_windows(mission, label, bin_indices):
    """2.56 s windows (1.28 s step). Sensor: TS_SENSOR real IMU samples chosen evenly by index inside the window;
    audio: TA_AUDIO real chunk timestamps chosen the same way (a window with fewer than 2 audio chunks — the
    recording edge — falls back to an even grid). tau_a / tau_s = those REAL timestamps relative to window start."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


class UAVWindowDataset(Dataset):
    """Windows -> tensors. Sensor channels are standardized with mean/std that must come from the TRAINING
    split (computed here only when none are passed, i.e. for the training set). tau is normalized by the
    window length."""

    def __init__(self, windows, sensor_mean=None, sensor_std=None):
        self.windows = windows
        if sensor_mean is None or sensor_std is None:
            all_s = np.concatenate([w["x_s"] for w in windows], axis=0)
            sensor_mean = all_s.mean(axis=0, keepdims=True)
            sensor_std = all_s.std(axis=0, keepdims=True) + SENSOR_STD_EPS
        self.sensor_mean, self.sensor_std = sensor_mean, sensor_std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        window_sec = float(w["window_sec"])
        return {
            "x_a": torch.tensor(w["x_a"], dtype=torch.float32),
            "x_s": torch.tensor((w["x_s"] - self.sensor_mean) / self.sensor_std, dtype=torch.float32),
            "tau_a": torch.tensor(w["tau_a"], dtype=torch.float32) / window_sec,
            "tau_s": torch.tensor(w["tau_s"], dtype=torch.float32) / window_sec,
            "label": torch.tensor(w["label"], dtype=torch.long),
        }


def split_fingerprint(windows):
    """(n windows, sum of raw sensor values, sum of sensor timestamps, class counts) — deterministic data check."""
    x_s = np.stack([w["x_s"] for w in windows]).astype(np.float64)
    tau_s = np.stack([w["tau_s"] for w in windows]).astype(np.float64)
    counts = tuple(np.bincount([w["label"] for w in windows], minlength=N_CLASSES).tolist())
    return (len(windows), round(float(x_s.sum()), 3), round(float(tau_s.sum()), 3), counts)


# --- 1. split + parse ---
mission_splits = split_missions(DATASET_ROOT)
split_dirs = {s: [m for label in sorted(mission_splits[s]) for m in mission_splits[s][label]] for s in mission_splits}
assert not set(split_dirs["train"]) & (set(split_dirs["val"]) | set(split_dirs["test"])), "LEAKAGE: mission in two splits"
parsed_missions = {m: parse_mission(m) for s in split_dirs for m in tqdm(split_dirs[s], desc=f"parse {s}", leave=False)}

# --- 2. strongest-64 bins from TRAINING audio only ---
bin_source_missions = [m for m in split_dirs["train"] if parsed_missions[m] is not None]
assert set(bin_source_missions) <= set(split_dirs["train"])
SELECTED_BINS = select_strongest_bins([parsed_missions[m] for m in bin_source_missions], N_AUDIO_BINS)
assert len(SELECTED_BINS) == N_AUDIO_BINS
print(f"[LEAKAGE CHECK] strongest-{N_AUDIO_BINS} bins computed from {len(bin_source_missions)} TRAINING missions only "
      f"(0 of {len(split_dirs['val']) + len(split_dirs['test'])} val/test missions) -> frozen for all splits: "
      f"{SELECTED_BINS.tolist()}")

# --- 3. windows (the same frozen bins for every split) ---
split_windows = {}
for split in ("train", "val", "test"):
    split_windows[split] = []
    for label in sorted(mission_splits[split]):
        for m in mission_splits[split][label]:
            if parsed_missions[m] is not None:
                split_windows[split].extend(slice_windows(parsed_missions[m], label, SELECTED_BINS))
    print(f"{split:<5}: {len(split_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
DATA_FINGERPRINT = {s: split_fingerprint(split_windows[s]) for s in split_windows}

# --- 4. datasets: sensor mean/std from the TRAINING split only, reused for val/test ---
train_ds = UAVWindowDataset(split_windows["train"])
val_ds = UAVWindowDataset(split_windows["val"], train_ds.sensor_mean, train_ds.sensor_std)
test_ds = UAVWindowDataset(split_windows["test"], train_ds.sensor_mean, train_ds.sensor_std)
_train_s = np.concatenate([w["x_s"] for w in split_windows["train"]], axis=0)
_val_s = np.concatenate([w["x_s"] for w in split_windows["val"]], axis=0)
assert val_ds.sensor_mean is train_ds.sensor_mean and test_ds.sensor_std is train_ds.sensor_std
assert np.array_equal(train_ds.sensor_mean, _train_s.mean(axis=0, keepdims=True))
print(f"[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | "
      f"train mean {np.round(train_ds.sensor_mean.ravel(), 3).tolist()} vs val's own mean "
      f"{np.round(_val_s.mean(axis=0), 3).tolist()} (not used)")
del _train_s, _val_s

# --- 5. real timestamps reach the model: tau must differ across windows ---
_tau_a = np.stack([train_ds[i]["tau_a"].numpy() for i in range(len(train_ds))])
_tau_s = np.stack([train_ds[i]["tau_s"].numpy() for i in range(len(train_ds))])
_n_unique_a, _n_unique_s = len(np.unique(_tau_a, axis=0)), len(np.unique(_tau_s, axis=0))
assert _n_unique_a > 1 and _n_unique_s > 1, "real timestamps are NOT reaching the model (tau identical for every window)"
assert not np.allclose(_tau_s, np.linspace(0.0, 1.0, TS_SENSOR)[None, :]), "tau_s equals a synthetic linspace grid"
print(f"[TIMESTAMP CHECK] {len(train_ds)} training windows: {_n_unique_a} distinct tau_a rows, {_n_unique_s} distinct "
      f"tau_s rows -> real per-window timestamps reach HTT/LACA ✅")
del _tau_a, _tau_s

Cloning https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git ...


Cloning into '/kaggle/working/tii_uav_dataset'...


Class 0 (19 flights) -> train 13, val 2, test 4
Class 1 (20 flights) -> train 14, val 3, test 3
Class 2 (20 flights) -> train 14, val 3, test 3
Class 3 (20 flights) -> train 14, val 3, test 3
Class 4 (20 flights) -> train 14, val 3, test 3


Updating files: 100% (217/217), done.


parse train:   0%|          | 0/69 [00:00<?, ?it/s]

parse val:   0%|          | 0/14 [00:00<?, ?it/s]

parse test:   0%|          | 0/16 [00:00<?, ?it/s]

[LEAKAGE CHECK] strongest-64 bins computed from 69 TRAINING missions only (0 of 30 val/test missions) -> frozen for all splits: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766]
train: 4934 windows | audio 64x64, sensor 128x6
val  : 954 windows | audio 64x64, sensor 128x6
test : 1130 windows | audio 64x64, sensor 128x6
[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | train mean [0.05299999937415123, -0.0430000014603138, -9.78600025177002, 0.004000000189989805, 0.003000000026077032, -0.0] vs val's own mean [0.029999999329447746, -0.061000000685453415, -9.748000144958496, 0.004999999888241291, 0.003000000026077032, -0.0] (not used)
[TIMESTAMP CHECK] 4934 training windows: 4934 distinct 

## 4. Model definition — two explicit switches
- **`HeterogeneousTemporalTokenization(d_model, time_enabled)`.** When ON: `norm(z + e_m + time_embedding(tau))`, which is v18 unchanged. When OFF: `norm(z + e_m)`. `time_embedding` is still built, so the RNG draws are the same, but it is never called.
- **`AeroSyncMamba(htt_time, laca_lag)`.** Sets the LACA scale to β = 1.0 (ON) or exactly 0.0 (OFF), which is the v18 β = 0 code path.

The v18 LayerNorm and zero-init of g are kept for the ON conditions. The warm-up and `set_beta` are removed: β never changes during a run. The cell checks that every condition has the same parameter count and prints where each switch acts.

In [4]:
# ==============================================================================
# MODEL — encoders -> HTT (modality + continuous-time embedding of the REAL tau)
# -> bi-directional LACA (lag-aware cross-attention; lag bias beta * LN_grid(g(Δτ)), beta = 1.0 when
#    LACA_LAG_BIAS_ENABLED else 0.0; 4 heads). v22: HTT's time embedding is switchable too.
# -> [H_a ; SEP ; H_s] -> 2 Mamba blocks -> attention pooling -> classifier.
# Auxiliary per-modality heads feed only the consistency loss.
# ==============================================================================
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding [+ time embedding(tau)]); modality 0 = audio, 1 = sensor.
    v22: time_enabled = HTT_TIME_EMBED_ENABLED. When False the time embedding is skipped entirely (no temporal
    position information enters through HTT); the module is still built so every condition draws the same RNG."""

    def __init__(self, d_model: int, time_enabled: bool = True):
        super().__init__()
        self.d_model = d_model
        self.time_enabled = time_enabled
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        if not self.time_enabled:
            return self.norm(z + e_m)                       # HTT_TIME_EMBED_ENABLED = False
        return self.norm(z + e_m + self.time_embedding(tau))


class GridLayerNorm(nn.Module):
    """LayerNorm for the lag-bias grid (v18).

    g(Δτ) produces ONE scalar per (query i, key j) pair, so its output is a (B, T_m, T_n) grid with no feature
    dimension > 1. A LayerNorm over a size-1 dimension would subtract each value from itself and return 0 for every
    input. Instead, the normalization is taken over each window's whole (T_m, T_n) grid, i.e. every bias value that
    enters that window's attention matrix. It is computed separately for each window: every window has its own real
    timestamps, and mixing windows would make one window's bias depend on the rest of the batch and differ between
    training and evaluation. There are no affine parameters, so beta is the only source of scale.

    Consequences: the subtracted mean is one constant per window, and softmax is unchanged by a constant added to
    every score, so only the division by the std affects attention. For an all-zero grid (the zero-init state) the
    output is exactly 0. As g develops variance v, the output std grows smoothly as sqrt(v / (v + eps)) toward 1."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (B, T_m, T_n)
        return F.layer_norm(x, x.shape[-2:], eps=self.eps)


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores (scaled by β)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        # v18 ZERO-INIT: g's final Linear starts at exactly 0 -> g(Δτ) = 0 for every Δτ and every seed. The layer was
        # already built (same RNG draws as v16), and nn.init.zeros_ draws no random numbers, so every other
        # layer's initial weights and the training RNG stream are unchanged.
        nn.init.zeros_(self.lag_bias[-1].weight)
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = GridLayerNorm()   # v18: no parameters

    def lag_bias_term(self, tau_m, tau_n):
        """LN_grid(g(Δτ)), BEFORE beta. Returns (B, 1, T_m, T_n), broadcast over the heads."""
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)   # (B, T_m, T_n, 1)
        g = self.lag_bias(delta_tau).squeeze(-1)                                # (B, T_m, T_n): one scalar per (i, j)
        return self.lag_bias_norm(g).unsqueeze(1)                               # normalized per window -> (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        lag_bias = self.lag_bias_term(tau_m, tau_n)

        attn = F.softmax(scores + self.beta * lag_bias, dim=-1)   # beta = 1.0 (lag bias ON) or exactly 0.0 (OFF)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, beta=beta)   # beta scales the lag bias in BOTH directions
        self.laca_sa = PairwiseLACA(d_model, beta=beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


class FallbackGatedConvBlock(nn.Module):
    """CPU-only stand-in when mamba_ssm is unavailable: depthwise causal conv + gated MLP. NOT a state-space
    model (no hidden state, no A/B/C, no discretization)."""

    def __init__(self, d_model: int, d_conv: int, expand: int):
        super().__init__()
        d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv, padding=d_conv - 1, groups=d_inner, bias=True)
        self.mixer = nn.Sequential(nn.Linear(d_inner, d_inner), nn.GELU(), nn.Linear(d_inner, d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        L = x.shape[1]
        x_branch, z_branch = self.in_proj(self.norm(x)).chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2))
        return x + self.out_proj(self.mixer(x_conv) * F.silu(z_branch))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if MAMBA_CLASS is not None:
        return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])
    return nn.ModuleList([FallbackGatedConvBlock(d_model, d_conv, expand) for _ in range(num_blocks)])


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, htt_time: bool = True, laca_lag: bool = False):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.htt_time_enabled = htt_time                  # HTT_TIME_EMBED_ENABLED
        self.laca_lag_enabled = laca_lag                  # LACA_LAG_BIAS_ENABLED
        self.laca_beta = LACA_BETA if laca_lag else 0.0   # OFF = exactly 0.0 (v18's beta = 0 code path)

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model, time_enabled=htt_time)
        self.laca = BiModalLACAFusion(d_model, beta=self.laca_beta)   # plain float: no params, no RNG draws
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model):
    """Finalized architecture, read from the constructed modules, including both v22 switches."""
    assert model.htt.time_enabled == model.htt_time_enabled
    beta = LACA_BETA if model.laca_lag_enabled else 0.0
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert laca.beta == beta, f"{name}: beta {laca.beta} != {beta} for LACA_LAG_BIAS_ENABLED={model.laca_lag_enabled}"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
        assert isinstance(laca.lag_bias_norm, GridLayerNorm) and not list(laca.lag_bias_norm.parameters())
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


_probe_counts = {}
for _c, (_h, _l) in CONDITIONS.items():
    _probe = AeroSyncMamba(htt_time=_h, laca_lag=_l)
    verify_architecture(_probe)
    _probe_counts[_c] = sum(p.numel() for p in _probe.parameters())
    del _probe
assert len(set(_probe_counts.values())) == 1, f"parameter count depends on the condition: {_probe_counts}"
_probe = AeroSyncMamba()
print(f"Model: {_probe_counts['A']:,} params in every condition {list(CONDITIONS)} (both modules always built) | fusion "
      f"blocks: {type(_probe.mamba_blocks[0]).__name__} x {len(_probe.mamba_blocks)} ({verify_architecture(_probe):,} "
      f"params) | LACA {LACA_NUM_HEADS} heads, g(Δτ) {LAG_BIAS_PARAMS} params | backbone: {FUSION_BACKBONE}")
del _probe
print("[SWITCHES] HTT_TIME_EMBED_ENABLED -> HeterogeneousTemporalTokenization.forward: ON norm(z + e_m + time_embedding(tau)) "
      "(v18, unchanged) | OFF norm(z + e_m)")
print(f"[SWITCHES] LACA_LAG_BIAS_ENABLED -> PairwiseLACA.forward: softmax(scores + beta * LN_grid(g(Δτ))) with beta = "
      f"{LACA_BETA} (ON) or exactly 0.0 (OFF, v18's beta = 0 path)")
print("[REUSE PRECONDITION] v18's HTT had no switch: its forward always added time_embedding(tau) -> v18 at beta = 0.0 "
      "IS Condition A (HTT-time ON, LACA-lag OFF).")


# --- LayerNorm placement: trace real tensor shapes through one LACA direction on 2 training windows ---
_ta = torch.stack([train_ds[i]["tau_a"] for i in range(2)])
_ts = torch.stack([train_ds[i]["tau_s"] for i in range(2)])
_probe = AeroSyncMamba(htt_time=True, laca_lag=True)
with torch.no_grad():
    for _name, _laca, _tm, _tn in (("laca_as", _probe.laca.laca_as, _ta, _ts), ("laca_sa", _probe.laca.laca_sa, _ts, _ta)):
        _dt = (_tm.unsqueeze(2) - _tn.unsqueeze(1)).unsqueeze(-1)
        _g = _laca.lag_bias(_dt).squeeze(-1)
        _out = _laca.lag_bias_term(_tm, _tn)
        print(f"[LAYERNORM] {_name}: Δτ {tuple(_dt.shape)} -> g(Δτ) {tuple(_g.shape)} = (B, T_m, T_n) -> "
              f"LN over dims {tuple(_g.shape[-2:])} ({_g.shape[-2] * _g.shape[-1]:,} values per window, per window) -> "
              f"{tuple(_out.shape)} -> x beta -> added to scores (B, {LACA_NUM_HEADS} heads, T_m, T_n)")
        assert float(_g.abs().max()) == 0.0 and float(_out.abs().max()) == 0.0, "zero-init g should give an all-zero term"
del _probe, _ta, _ts
print("[LAYERNORM] placement: PairwiseLACA.forward -> scores + beta * LN_grid(g(Δτ)); the LayerNorm is applied to g's "
      "scalar output BEFORE beta.")
print("[LAYERNORM] why this shape: g outputs one scalar per (i, j) pair, so there is no feature dim > 1 (LN over a "
      "size-1 dim would output 0 for every input). Normalizing over each window's full (T_m, T_n) grid covers every "
      "bias value in that window's attention matrix. It is per window, never across the batch, so a window's bias "
      "does not depend on its batch-mates and is the same in train and eval. No affine: beta is the only scale. The "
      "subtracted mean is one constant per window, which softmax ignores; only the std normalization affects "
      "attention.")

Model: 1,688,788 params in every condition ['A', 'B', 'C', 'D'] (both modules always built) | fusion blocks: MambaBlock x 2 (874,496 params) | LACA 4 heads, g(Δτ) 97 params | backbone: mamba_ssm.Mamba (real selective-scan SSM, CUDA)
[SWITCHES] HTT_TIME_EMBED_ENABLED -> HeterogeneousTemporalTokenization.forward: ON norm(z + e_m + time_embedding(tau)) (v18, unchanged) | OFF norm(z + e_m)
[SWITCHES] LACA_LAG_BIAS_ENABLED -> PairwiseLACA.forward: softmax(scores + beta * LN_grid(g(Δτ))) with beta = 1.0 (ON) or exactly 0.0 (OFF, v18's beta = 0 path)
[REUSE PRECONDITION] v18's HTT had no switch: its forward always added time_embedding(tau) -> v18 at beta = 0.0 IS Condition A (HTT-time ON, LACA-lag OFF).
[LAYERNORM] laca_as: Δτ (2, 64, 128, 1) -> g(Δτ) (2, 64, 128) = (B, T_m, T_n) -> LN over dims (64, 128) (8,192 values per window, per window) -> (2, 1, 64, 128) -> x beta -> added to scores (B, 4 heads, T_m, T_n)
[LAYERNORM] laca_sa: Δτ (2, 128, 64, 1) -> g(Δτ) (2, 128, 64) = (B, T_m, T_n) -> 

## 5. Metrics, switch checks, H2 sweep and the training function
**Evaluation.** `evaluate_probs` returns the 4 standard metrics, the full confusion matrix, Class 3 and Class 4 recall, and the 3→4 and 4→3 counts.

**Switch checks** (on a fixed real batch, before training and at the best checkpoint):
- `htt_time_contribution` measures the time embedding's actual effect: max|HTT(z, τ) − norm(z + e_m)| and max|HTT(z, τ) − HTT(z, τ + 0.3)|. Both must be **exactly 0** when HTT time is OFF, and above 0 when ON.
- `lag_bias_contribution` (from v18) measures max|β · LN(g(Δτ))|. It is asserted **exactly 0 at every epoch** when LACA lag is OFF. When ON it starts at 0, because g is zero-initialized, and must be above 0 by the end.

**`h2_sweep`** is the v4/v5 H2 procedure: sensor timestamps become clamp(τ_s + δ, 0, 1) at the model input, and macro-F1 retention is reported relative to δ = 0.

**`train_one_run(seed, condition)`:**
1. Resets every RNG and builds the model with the condition's switches.
2. Checks that its initial weights and RNG state equal the v18 run for that seed. They are the same in all 4 conditions.
3. Trains, then evaluates the best checkpoint.

In [5]:
# ==============================================================================
# TRAINING — loss (unchanged), metrics (4 standard + confusion matrix + Class 3/4
# recall + 3->4 / 4->3), switch checks (HTT time term, LACA lag term), the H2
# temporal-shift sweep, and train_one_run(seed, condition).
# ==============================================================================
class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}


def compute_metrics(y_true, y_prob):
    """accuracy, macro-F1, balanced accuracy, macro one-vs-rest AUROC from class probabilities (prediction = argmax)."""
    y_pred = y_prob.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_auroc": float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")),
    }


def train_class_weights(windows):
    """Inverse class frequency over the TRAINING windows."""
    counts = np.bincount([w["label"] for w in windows], minlength=N_CLASSES)
    return torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32)


def _batch_to_device(batch, non_blocking=False):
    return tuple(batch[k].to(device, non_blocking=non_blocking) for k in ("x_a", "x_s", "tau_a", "tau_s", "label"))


def predict_probs(model, loader):
    """Softmax probabilities and true labels over a loader (eval mode, no grad), in loader order."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits, _, _, _ = model(xa, xs, ta, ts)
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
CLASS_WEIGHTS = train_class_weights(split_windows["train"]).to(device)
TEST_LABELS = np.array([w["label"] for w in split_windows["test"]], dtype=np.int64)   # test-window order, fixed


# Fixed probe batch for the lag-contribution check: timestamps of the first BATCH_SIZE training windows, read by
# index (no DataLoader, no RNG). The lag-bias term depends only on the timestamps.
LAG_PROBE_TAUS = tuple(torch.stack([train_ds[i][k] for i in range(min(BATCH_SIZE, len(train_ds)))])
                       for k in ("tau_a", "tau_s"))


def lag_bias_contribution(model):
    """Per LACA direction: max|beta * LN_grid(g(Δτ))|, the exact term PairwiseLACA adds to the attention scores
    (computed by the same lag_bias_term method, with the beta currently in effect), plus the raw std of g before
    the LayerNorm. g has no dropout or BatchNorm: this uses no random numbers and changes no state."""
    ta, ts = (t.to(device) for t in LAG_PROBE_TAUS)
    out = {}
    with torch.no_grad():
        for name, laca, tau_m, tau_n in (("laca_as", model.laca.laca_as, ta, ts), ("laca_sa", model.laca.laca_sa, ts, ta)):
            term = laca.lag_bias_term(tau_m, tau_n)
            contrib = laca.beta * term
            g_raw = laca.lag_bias((tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)).squeeze(-1)
            out[name] = {"beta": laca.beta, "contrib_absmax": float(contrib.abs().max()),
                         "contrib_finite": bool(torch.isfinite(contrib).all()), "g_raw_std": float(g_raw.std())}
    return out


def g_final_layer_absmax(model):
    """max|weight| and max|bias| of g's final Linear(32, 1), per LACA direction."""
    return {name: (float(getattr(model.laca, name).lag_bias[-1].weight.detach().abs().max()),
                   float(getattr(model.laca, name).lag_bias[-1].bias.detach().abs().max()))
            for name in ("laca_as", "laca_sa")}


from sklearn.metrics import confusion_matrix


def evaluate_probs(y_true, y_prob):
    """4 standard metrics + full 5x5 confusion matrix + per-class recall + Class 3/4 recall + 3->4 / 4->3 counts."""
    cm = confusion_matrix(y_true, y_prob.argmax(axis=1), labels=list(range(N_CLASSES)))
    recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    return {**compute_metrics(y_true, y_prob), "confusion_matrix": cm,
            **{f"recall_c{c}": float(recall[c]) for c in range(N_CLASSES)},
            "class3_recall": float(recall[3]), "class4_recall": float(recall[4]),
            "err_3to4": int(cm[3, 4]), "err_4to3": int(cm[4, 3])}


def evaluate_cm(cm, metrics):
    """Same fields from a stored confusion matrix (reused runs); the 4 metrics come from the stored values."""
    cm = np.asarray(cm)
    recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    return {**{m: metrics[m] for m in METRICS}, "confusion_matrix": cm,
            **{f"recall_c{c}": float(recall[c]) for c in range(N_CLASSES)},
            "class3_recall": float(recall[3]), "class4_recall": float(recall[4]),
            "err_3to4": int(cm[3, 4]), "err_4to3": int(cm[4, 3])}


def print_confusion(ev, tag):
    cm = ev["confusion_matrix"]
    print(f"[CONFUSION {tag}] rows = TRUE severity, cols = PREDICTED severity (test set, {int(cm.sum())} windows)")
    print("            " + "".join(f"pred {j:>2} " for j in range(N_CLASSES)) + "|  recall")
    for i in range(N_CLASSES):
        print(f"   true {i:>2} " + "".join(f"{int(cm[i, j]):>8}" for j in range(N_CLASSES)) + f"  |  {ev[f'recall_c{i}']:.4f}")
    print(f"[CLASS 3/4 {tag}] Class 3 recall {ev['class3_recall']:.4f} | Class 4 recall {ev['class4_recall']:.4f} | "
          f"3->4 (true 3, pred 4) {ev['err_3to4']} | 4->3 (true 4, pred 3) {ev['err_4to3']}")


METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"]
SWITCH_PROBE_IDX = np.linspace(0, len(train_ds) - 1, BATCH_SIZE).astype(int)   # fixed real batch, read by index


def htt_time_contribution(model):
    """What HTT's time embedding actually contributes, on a fixed real batch (eval mode, RNG untouched):
    term = max|HTT(z, tau) - norm(z + e_m)| and tau_sens = max|HTT(z, tau) - HTT(z, tau + 0.3)|, per modality."""
    items = [train_ds[int(i)] for i in SWITCH_PROBE_IDX]
    xa, xs, ta, ts = (torch.stack([it[k] for it in items]).to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
    was_training = model.training
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    out = {}
    with torch.random.fork_rng(devices=cuda_devices), torch.no_grad():
        model.eval()
        za = model.audio_encoder(xa)
        zs = model.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        for name, z, tau, idx in (("audio", za, ta, 0), ("sensor", zs, ts, 1)):
            e_m = model.htt.modality_embeddings(torch.tensor(idx, device=device)).view(1, 1, model.d_model)
            actual = model.htt(z, tau, modality_idx=idx)
            out[name] = {"term": float((actual - model.htt.norm(z + e_m)).abs().max()),
                         "tau_sens": float((actual - model.htt(z, tau + 0.3, modality_idx=idx)).abs().max())}
    model.train(was_training)
    return out


def check_switches(model, tag, when, final=False):
    """Print and enforce both switches: a disabled contribution must be exactly 0."""
    h = htt_time_contribution(model)
    lag = lag_bias_contribution(model)
    lag_max = max(r["contrib_absmax"] for r in lag.values())
    h_max = max(max(v["term"], v["tau_sens"]) for v in h.values())
    print(f"{tag} [SWITCH CHECK {when}] HTT_TIME_EMBED_ENABLED={model.htt.time_enabled}: time-embedding term max "
          + ", ".join(f"{k} {v['term']:.3e}" for k, v in h.items()) + " | output change for tau+0.3: "
          + ", ".join(f"{k} {v['tau_sens']:.3e}" for k, v in h.items()))
    print(f"{tag} [SWITCH CHECK {when}] LACA_LAG_BIAS_ENABLED={model.laca_lag_enabled} (beta read back: "
          f"{model.laca.laca_as.beta} / {model.laca.laca_sa.beta}): max|beta * LN(g(Δτ))| = {lag_max!r}")
    if not model.htt_time_enabled:
        assert h_max == 0.0, f"HTT time embedding is OFF but contributes {h_max!r}"
        print(f"{tag} [SWITCH CHECK {when}] ✅ HTT time OFF: its contribution is EXACTLY 0 and HTT ignores tau entirely")
    else:
        assert h_max > 0.0, "HTT time embedding is ON but contributes nothing"
        print(f"{tag} [SWITCH CHECK {when}] ✅ HTT time ON: the time embedding changes the tokens and responds to tau")
    if not model.laca_lag_enabled:
        assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), f"LACA lag OFF but {lag}"
        print(f"{tag} [SWITCH CHECK {when}] ✅ LACA lag OFF: the lag term added to the attention scores is EXACTLY 0")
    elif final and lag_max == 0.0:
        print(f"🚨 {tag} LACA lag is ON but its contribution is still 0 after training — the switch is not working")
    elif lag_max == 0.0:
        print(f"{tag} [SWITCH CHECK {when}] LACA lag ON (beta {LACA_BETA}); the term is 0 only because g's final layer is "
              f"zero-initialized (v18 form) — it grows from the first update")
    else:
        print(f"{tag} [SWITCH CHECK {when}] ✅ LACA lag ON: the lag term is active ({lag_max:.4f})")
    return h, lag_max


def predict_probs_shift(model, loader, delta):
    """H2 (v4/v5): sensor timestamps -> clamp(tau_s + delta, 0, 1) at the model input (HTT and LACA both see it)."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits = model(xa, xs, ta, torch.clamp(ts + delta, 0.0, 1.0))[0]
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


def h2_sweep(model, cond, seed, clean_ev):
    """Macro-F1 retention under artificial audio <-> IMU delays; returns one row per offset."""
    tag = f"[H2 {cond} seed={seed}]"
    rows, base_f1, base_probs = [], None, None
    print(f"{tag} temporal-shift robustness: tau_s <- clamp(tau_s + delta, 0, 1)")
    print(f"      {'offset':>8} {'accuracy':>9} {'macro-F1':>9} {'AUROC':>8} {'C4 recall':>10} {'retention':>10}  predictions")
    for delta in H2_OFFSETS:
        probs, labels = predict_probs_shift(model, test_loader, delta)
        ev = evaluate_probs(labels, probs)
        if delta == 0.0:
            base_f1, base_probs = ev["macro_f1"], probs
            clean_match = max(abs(ev[m] - clean_ev[m]) for m in METRICS)
        same = np.array_equal(probs, base_probs)
        rows.append({"condition": cond, "seed": seed, "offset": delta, "accuracy": ev["accuracy"], "macro_f1": ev["macro_f1"],
                     "macro_auroc": ev["macro_auroc"], "class4_recall": ev["class4_recall"],
                     "retention_pct": 100.0 * ev["macro_f1"] / base_f1, "identical_to_offset0": same})
        print(f"      {'+' + format(delta, '.2f'):>8} {ev['accuracy']:>9.4f} {ev['macro_f1']:>9.4f} {ev['macro_auroc']:>8.4f} "
              f"{ev['class4_recall']:>10.4f} {rows[-1]['retention_pct']:>9.2f}%  {'identical to +0.00' if same else 'changed'}")
    print(f"{tag} offset 0 (with the clamp) vs the clean test evaluation: max |delta| {clean_match:.1e}"
          + (" ✅" if clean_match < 1e-9 else " ⚠️ the clamp alters some timestamps even at offset 0"))
    if not CONDITIONS[cond][0] and not CONDITIONS[cond][1]:
        flat = all(r["identical_to_offset0"] for r in rows)
        print(f"{tag} Condition D uses no timestamps anywhere -> the curve must be exactly flat: "
              + ("✅ every offset gives bit-identical predictions" if flat else "🚨 NOT flat — a timestamp path is still active"))
    return rows


def train_one_run(seed: int, cond: str):
    """Train one model of `cond` with `seed` applied to every RNG, evaluate it on the test set.
    Returns (trained model on CPU, result dict incl. test probabilities and confusion matrix)."""
    t0 = time.time()
    htt_on, laca_on = CONDITIONS[cond]
    tag = f"[{cond} seed={seed} HTT-time {'ON' if htt_on else 'OFF'} / LACA-lag {'ON' if laca_on else 'OFF'}]"
    set_all_seeds(seed)
    model = AeroSyncMamba(htt_time=htt_on, laca_lag=laca_on).to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()
    rng_sha1 = hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()
    mamba_params = verify_architecture(model)
    ref = V18_BETA0_RUNS[seed]
    paired_ok = init_sha1[:12] == ref["init_sha1"] and rng_sha1[:12] == ref["rng_sha1"]
    print(f"{tag} HTT_TIME_EMBED_ENABLED = {model.htt.time_enabled} | LACA_LAG_BIAS_ENABLED = {model.laca_lag_enabled} "
          f"(beta {model.laca.laca_as.beta}) — read from the constructed model | fusion {type(model.mamba_blocks[0]).__name__} "
          f"({mamba_params:,} params)")
    print(f"{tag} [PAIRING] initial weights {init_sha1[:12]} vs v18 {ref['init_sha1']} | RNG {rng_sha1[:12]} vs v18 "
          f"{ref['rng_sha1']} -> " + ("✅ identical start and RNG stream (same in all 4 conditions)" if paired_ok else
                                      "🚨 MISMATCH — this seed's conditions are not cleanly paired"))
    check_switches(model, tag, "before training")

    criterion = AeroSyncLoss(CLASS_WEIGHTS)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

    best_val_f1, best_state, best_epoch = 0.0, None, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = defaultdict(float)
        for batch in tqdm(train_loader, desc=f"{cond} seed {seed} epoch {epoch:02d}/{EPOCHS}", leave=False):
            xa, xs, ta, ts, y = _batch_to_device(batch, non_blocking=True)
            optimizer.zero_grad()
            logits, pooled, aux_logits, _ = model(xa, xs, ta, ts)
            loss, parts = criterion(logits, y, pooled, aux_logits)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running["loss"] += loss.item()
            for k, v in parts.items():
                running[k] += v
        scheduler.step()

        lag = lag_bias_contribution(model)
        lag_max = max(r["contrib_absmax"] for r in lag.values())
        if not laca_on:
            assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), f"lag leak at epoch {epoch}"

        val_probs, val_labels = predict_probs(model, val_loader)
        val_f1 = compute_metrics(val_labels, val_probs)["macro_f1"]
        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1, best_state, best_epoch = val_f1, copy.deepcopy(model.state_dict()), epoch
        n = len(train_loader)
        print(f"{tag} epoch {epoch:02d}/{EPOCHS} | lag term {lag_max:.4f}{' (exactly 0 ✅)' if not laca_on else ''} | train "
              f"loss {running['loss'] / n:.4f} (cls {running['loss_cls'] / n:.4f}, align {running['loss_align'] / n:.4f}, "
              f"cons {running['loss_cons'] / n:.4f}) | val macro-F1 {val_f1:.4f}{'  * best' if improved else ''}")

    model.load_state_dict(best_state)
    h_final, lag_final = check_switches(model, tag, f"best checkpoint, epoch {best_epoch}", final=True)
    test_probs, test_labels = predict_probs(model, test_loader)
    ev = evaluate_probs(test_labels, test_probs)
    run_name = f"{cond}_seed{seed}"
    torch.save(best_state, CHECKPOINT_DIR / f"{run_name}_best.pt")
    np.save(PROBS_DIR / f"{run_name}_test_probs.npy", test_probs)
    np.save(PROBS_DIR / f"{run_name}_test_labels.npy", test_labels)

    result = {
        "condition": cond, "seed": seed, "htt_time": htt_on, "laca_lag": laca_on,
        **ev,
        "best_val_macro_f1": round(best_val_f1, 6),
        "best_epoch": best_epoch,
        "n_test_windows": len(test_labels),
        "init_sha1": init_sha1[:12],
        "rng_sha1": rng_sha1[:12],
        "paired_with_v18": paired_ok,
        "htt_time_term_final": max(v["term"] for v in h_final.values()),
        "lag_term_final": lag_final,
        "train_seconds": round(time.time() - t0, 1),
        "test_probs": test_probs,
        "test_labels": test_labels,
    }
    return model, result

## 6. Driver — reuse decision for Condition A, then 16 (or 20) runs grouped by seed
**Order.** Seed by seed: A, B, C, D.

**Seed 42.** All four conditions are trained. Condition A seed 42 is compared with v18's β = 0 seed-42 run:
- **If it reproduces v18** (every metric within ±0.0005), Condition A is **reused** for the other seeds. v18's 4 metrics plus v20's confusion matrices for the same runs are logged with `source = reused v18`.
- **Otherwise** Condition A is trained for every seed.

**Seed 42 also gets the H2 sweep** right after each condition's training, on the same model in memory.

**After every run** the driver prints the full confusion matrix, appends a row to `temporal_2x2_ablation_results.csv`, frees the GPU and prints progress. H2 rows go to `temporal_2x2_h2_robustness.csv`.

In [6]:
# ==============================================================================
# DRIVER — seed by seed: A, B, C, D. Condition A is retrained for seed 42 to
# verify the v18 reuse; if it reproduces v18, seeds 123/2024/7/99 reuse v18's
# rows, otherwise Condition A is retrained everywhere. H2 sweep on seed 42.
# ==============================================================================
import pandas as pd

for f in (RESULTS_CSV, H2_CSV):
    if f.exists():
        f.rename(f.with_suffix(f".{datetime.now():%Y%m%d_%H%M%S}.bak.csv"))
RESULTS_CSV_FIELDS = (["slot", "condition", "seed", "htt_time", "laca_lag", "source"] + METRICS
                      + ["class3_recall", "class4_recall", "err_3to4", "err_4to3"] + [f"recall_c{c}" for c in range(N_CLASSES)]
                      + ["confusion_matrix", "best_val_macro_f1", "best_epoch", "init_sha1", "rng_sha1", "paired_with_v18",
                         "htt_time_term_final", "lag_term_final", "train_seconds", "finished_at"])


def log_row(slot, result, source):
    write_header = not RESULTS_CSV.exists()
    row = {k: result[k] for k in RESULTS_CSV_FIELDS if k in result}
    row.update(slot=slot, source=source, confusion_matrix=json.dumps(np.asarray(result["confusion_matrix"]).tolist()),
               finished_at=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    with open(RESULTS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_CSV_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def log_h2(rows):
    write_header = not H2_CSV.exists()
    with open(H2_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        if write_header:
            writer.writeheader()
        writer.writerows(rows)


# --- the stored v18 runs: confusion matrices must reproduce v18's metrics exactly ---
for seed in SEEDS:
    r = V18_BETA0_RUNS[seed]
    cm = np.array(r["confusion_matrix"])
    y_true = np.repeat(np.arange(N_CLASSES), cm.sum(axis=1))
    y_pred = np.concatenate([np.repeat(np.arange(N_CLASSES), cm[i]) for i in range(N_CLASSES)])
    assert np.array_equal(np.bincount(y_true, minlength=N_CLASSES), np.bincount(TEST_LABELS, minlength=N_CLASSES))
    worst = max(abs(accuracy_score(y_true, y_pred) - r["accuracy"]), abs(f1_score(y_true, y_pred, average="macro") - r["macro_f1"]),
                abs(balanced_accuracy_score(y_true, y_pred) - r["balanced_accuracy"]))
    assert worst < 1e-12, f"stored confusion matrix for seed {seed} does not reproduce v18"
print("[CONDITION A STORE] v18 beta=0.0 runs: confusion matrices (from v20's identical regeneration) reproduce v18's "
      "accuracy / macro-F1 / balanced accuracy exactly for all 5 seeds ✅ | macro-F1 "
      + ", ".join(f"{s}: {V18_BETA0_RUNS[s]['macro_f1']:.4f}" for s in SEEDS))

REUSE_A, REUSE_A_DELTA = None, None
run_results, h2_results, sources = {}, {}, {}
n_trained = 0
t_driver = time.time()
for slot, (seed, cond) in enumerate(RUN_ORDER, start=1):
    print("\n" + "=" * 100 + f"\nSLOT {slot}/{len(RUN_ORDER)} — seed={seed}, condition {cond} (HTT-time "
          f"{'ON' if CONDITIONS[cond][0] else 'OFF'}, LACA-lag {'ON' if CONDITIONS[cond][1] else 'OFF'})\n" + "=" * 100)
    if cond == "A" and seed != REUSE_CHECK_SEED and REUSE_A:
        ref = V18_BETA0_RUNS[seed]
        result = {"condition": "A", "seed": seed, "htt_time": True, "laca_lag": False,
                  **evaluate_cm(ref["confusion_matrix"], ref), "init_sha1": ref["init_sha1"], "rng_sha1": ref["rng_sha1"]}
        run_results[(seed, cond)] = result
        sources[(seed, cond)] = "reused v18 beta=0.0 (not retrained)"
        print(f"[A seed={seed}] REUSED v18 beta=0.0 run — NOT retrained (seed-{REUSE_CHECK_SEED} verification passed): "
              f"accuracy {ref['accuracy']:.4f} | macro-F1 {ref['macro_f1']:.4f} | balanced accuracy "
              f"{ref['balanced_accuracy']:.4f} | macro-AUROC {ref['macro_auroc']:.4f}")
        print_confusion(result, f"A seed={seed} (reused)")
        log_row(slot, result, sources[(seed, cond)])
    else:
        model, result = train_one_run(seed, cond)
        n_trained += 1
        run_results[(seed, cond)] = result
        sources[(seed, cond)] = "trained in v22"
        print(f"[{cond} seed={seed}] TEST: accuracy {result['accuracy']:.4f} | macro-F1 {result['macro_f1']:.4f} | "
              f"balanced accuracy {result['balanced_accuracy']:.4f} | macro-AUROC {result['macro_auroc']:.4f} | best epoch "
              f"{result['best_epoch']} | {result['train_seconds']:.0f}s")
        print_confusion(result, f"{cond} seed={seed}")
        if seed == H2_SEED:
            h2_results[cond] = h2_sweep(model, cond, seed, result)
            log_h2(h2_results[cond])
        if cond == "A" and seed == REUSE_CHECK_SEED:
            ref = V18_BETA0_RUNS[seed]
            REUSE_A_DELTA = max(abs(result[m] - ref[m]) for m in METRICS)
            REUSE_A = REUSE_A_DELTA <= SAME_SEED_TOLERANCE and result["paired_with_v18"]
            print("\n" + "#" * 100)
            print(f"#  CONDITION A REUSE DECISION — seed {seed} retrained vs v18 beta=0.0 seed {seed}:")
            for m in METRICS:
                print(f"#    {m:<18} v18 {ref[m]:.4f} | v22 Condition A {result[m]:.4f} | delta {result[m] - ref[m]:+.4f}")
            if REUSE_A:
                print(f"#  ✅ REPRODUCES v18 (max |delta| {REUSE_A_DELTA:.1e} <= {SAME_SEED_TOLERANCE}) -> Condition A for seeds "
                      f"{[s for s in SEEDS if s != seed]} is REUSED from v18 (not retrained).")
            else:
                print(f"#  🚨 does NOT reproduce v18 (max |delta| {REUSE_A_DELTA:.4f}) -> there is doubt, so Condition A will "
                      f"be RETRAINED for every seed instead of reused.")
            print("#" * 100)
        log_row(slot, result, sources[(seed, cond)])
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    elapsed = time.time() - t_driver
    remaining_trained = sum(1 for s, c in RUN_ORDER[slot:] if not (c == "A" and s != REUSE_CHECK_SEED and REUSE_A))
    print(f"\n>>> {slot}/{len(RUN_ORDER)} slots complete: seed={seed}, condition {cond} ({sources[(seed, cond)]}) | "
          f"{n_trained} trained so far | elapsed {elapsed / 60:.1f} min | est. remaining "
          f"{elapsed / max(n_trained, 1) * remaining_trained / 60:.1f} min ({remaining_trained} runs to train)")

print(f"\nAll {len(RUN_ORDER)} slots done: {n_trained} trained, {len(RUN_ORDER) - n_trained} reused, in "
      f"{(time.time() - t_driver) / 60:.1f} min. Results: {RESULTS_CSV} | H2: {H2_CSV}")

[CONDITION A STORE] v18 beta=0.0 runs: confusion matrices (from v20's identical regeneration) reproduce v18's accuracy / macro-F1 / balanced accuracy exactly for all 5 seeds ✅ | macro-F1 42: 0.9057, 123: 0.8827, 2024: 0.8764, 7: 0.9019, 99: 0.8638

SLOT 1/20 — seed=42, condition A (HTT-time ON, LACA-lag OFF)
[A seed=42 HTT-time ON / LACA-lag OFF] HTT_TIME_EMBED_ENABLED = True | LACA_LAG_BIAS_ENABLED = False (beta 0.0) — read from the constructed model | fusion MambaBlock (874,496 params)
[A seed=42 HTT-time ON / LACA-lag OFF] [PAIRING] initial weights 5850841e699f vs v18 5850841e699f | RNG 5641b401720d vs v18 5641b401720d -> ✅ identical start and RNG stream (same in all 4 conditions)
[A seed=42 HTT-time ON / LACA-lag OFF] [SWITCH CHECK before training] HTT_TIME_EMBED_ENABLED=True: time-embedding term max audio 5.743e-01, sensor 6.356e-01 | output change for tau+0.3: audio 7.920e-01, sensor 8.713e-01
[A seed=42 HTT-time ON / LACA-lag OFF] [SWITCH CHECK before training] LACA_LAG_BIAS_ENA

A seed 42 epoch 01/20:   0%|          | 0/155 [00:00<?, ?it/s]

[A seed=42 HTT-time ON / LACA-lag OFF] epoch 01/20 | lag term 0.0000 (exactly 0 ✅) | train loss 1.4174 (cls 1.0707, align 3.1025, cons 0.3650) | val macro-F1 0.8261  * best


A seed 42 epoch 02/20:   0%|          | 0/155 [00:00<?, ?it/s]

[A seed=42 HTT-time ON / LACA-lag OFF] epoch 02/20 | lag term 0.0000 (exactly 0 ✅) | train loss 0.9259 (cls 0.5977, align 2.6650, cons 0.6171) | val macro-F1 0.6814


...

[D seed=99 HTT-time OFF / LACA-lag OFF] epoch 18/20 | lag term 0.0000 (exactly 0 ✅) | train loss 0.3446 (cls 0.1764, align 1.2536, cons 0.4287) | val macro-F1 0.8553


D seed 99 epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

[D seed=99 HTT-time OFF / LACA-lag OFF] epoch 19/20 | lag term 0.0000 (exactly 0 ✅) | train loss 0.3119 (cls 0.1435, align 1.2520, cons 0.4318) | val macro-F1 0.8896


D seed 99 epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

[D seed=99 HTT-time OFF / LACA-lag OFF] epoch 20/20 | lag term 0.0000 (exactly 0 ✅) | train loss 0.3163 (cls 0.1487, align 1.2431, cons 0.4326) | val macro-F1 0.8921
[D seed=99 HTT-time OFF / LACA-lag OFF] [SWITCH CHECK best checkpoint, epoch 16] HTT_TIME_EMBED_ENABLED=False: time-embedding term max audio 0.000e+00, sensor 0.000e+00 | output change for tau+0.3: audio 0.000e+00, sensor 0.000e+00
[D seed=99 HTT-time OFF / LACA-lag OFF] [SWITCH CHECK best checkpoint, epoch 16] LACA_LAG_BIAS_ENABLED=False (beta read back: 0.0 / 0.0): max|beta * LN(g(Δτ))| = 0.0
[D seed=99 HTT-time OFF / LACA-lag OFF] [SWITCH CHECK best checkpoint, epoch 16] ✅ HTT time OFF: its contribution is EXACTLY 0 and HTT ignores tau entirely
[D seed=99 HTT-time OFF / LACA-lag OFF] [SWITCH CHECK best checkpoint, epoch 16] ✅ LACA lag OFF: the lag term added to the attention scores is EXACTLY 0
[D seed=99] TEST: accuracy 0.8761 | macro-F1 0.8706 | balanced accuracy 0.8675 | macro-AUROC 0.9850 | best epoch 16 | 230s
[CON

## 7. Three paired questions, the H2 curves, and the answer
For each question, the difference is taken per seed (the first condition minus the second), for **macro-F1** and for **Class 4 recall**:
- **Q1. Does removing HTT's time embedding hurt?** A − D (positive means HTT time helps).
- **Q2. Does LACA's lag bias add anything beyond HTT?** C − A.
- **Q3. Can LACA's lag bias work without HTT?** B − D.

Each gets the mean paired difference, the std of the paired differences, the win count, a **paired t-test (primary)** and a Wilcoxon signed-rank test (secondary).

**Why the t-test is primary.** With 5 pairs, the exact two-sided Wilcoxon p-value can never go below 2/2⁵ = 0.0625, so it can never reach 0.05. Class 4 recall moves in steps of 1/201, which makes ties likely. The t-test assumes roughly normal differences, which 5 pairs can't check, so the Wilcoxon test is reported alongside it.

**When an effect counts as "real".** The project's usual rule: it wins on at least 4 of 5 seeds, **and** its mean is larger than the std of its differences or p < 0.05. Whether p < 0.05 is also stated separately. The interaction (C − A) − (B − D) is printed as a description only.

**Answer.** The final block states the answer to each question and which pattern the evidence supports: (i) temporal information not shown to help; (ii) HTT captures it and LACA is redundant; (iii) LACA helps only together with HTT; or (iv) another pattern, described plainly.

In [7]:
# ==============================================================================
# ANALYSIS — per-condition table, three paired questions (macro-F1 and Class 4
# recall), interaction, H2 retention curves, and the explicit answer.
# ==============================================================================
from scipy import stats

df = pd.read_csv(RESULTS_CSV)
tab = {(int(r["seed"]), r["condition"]): r for _, r in df.iterrows()}
complete = [s for s in SEEDS if all((s, c) in tab for c in CONDITIONS)]
n = len(complete)
print(f"Complete seed groups (all 4 conditions): {n}/{len(SEEDS)} -> {complete}")
if n < len(SEEDS):
    print(f"⚠️  PARTIAL RESULT: statistics use the {n} complete seed groups only.")
assert n >= 2, "need at least 2 complete seed groups for any paired statistic"

# --- per-condition means ---
print("\n" + "=" * 118)
print(f"PER-CONDITION MEANS over seeds {complete}")
print("=" * 118)
print(f"  {'condition':<34} {'accuracy':>9} {'macro-F1':>9} {'bal-acc':>8} {'AUROC':>8} {'C3 rec':>7} {'C4 rec':>7} "
      f"{'3->4':>6} {'4->3':>6}")
for c, (h, l) in CONDITIONS.items():
    g = lambda k: np.mean([float(tab[(s, c)][k]) for s in complete])
    print(f"  {c + ' (HTT-time ' + ('ON' if h else 'OFF') + ', LACA-lag ' + ('ON' if l else 'OFF') + ')':<34} "
          f"{g('accuracy'):>9.4f} {g('macro_f1'):>9.4f} {g('balanced_accuracy'):>8.4f} {g('macro_auroc'):>8.4f} "
          f"{g('class3_recall'):>7.4f} {g('class4_recall'):>7.4f} {g('err_3to4'):>6.1f} {g('err_4to3'):>6.1f}")


def paired_stats(new, base):
    d = np.asarray(new, dtype=float) - np.asarray(base, dtype=float)
    k = len(d)
    out = {"d": d, "n": k, "mean": float(d.mean()), "sd": float(d.std(ddof=1)), "wins": int((d > 0).sum()),
           "losses": int((d < 0).sum()), "ties": int((d == 0).sum())}
    out["p_t"] = float(stats.ttest_rel(new, base).pvalue) if out["sd"] > 0 else float("nan")
    try:
        out["p_w"] = float(stats.wilcoxon(d, alternative="two-sided", method="exact").pvalue)
    except Exception:
        try:
            out["p_w"] = float(stats.wilcoxon(d, alternative="two-sided").pvalue)
        except Exception:
            out["p_w"] = float("nan")
    return out


def effect_verdict(P):
    if P["mean"] > 0 and P["wins"] >= P["n"] - 1 and (P["mean"] > P["sd"] or P["p_t"] < PAIRED_ALPHA):
        return "REAL POSITIVE"
    if P["mean"] < 0 and P["losses"] >= P["n"] - 1 and (-P["mean"] > P["sd"] or P["p_t"] < PAIRED_ALPHA):
        return "REAL NEGATIVE"
    return "NOT ESTABLISHED"


QUESTIONS = [("Q1", "Does removing HTT's time embedding hurt?", "A", "D", "HTT time embedding (LACA off in both)"),
             ("Q2", "Does LACA's lag bias add benefit beyond HTT alone?", "C", "A", "LACA lag bias on top of HTT"),
             ("Q3", "Can LACA's lag bias work without HTT?", "B", "D", "LACA lag bias with no HTT time")]
Q = {}
for qid, question, new, base, what in QUESTIONS:
    print("\n" + "=" * 118)
    print(f"{qid}. {question}  ->  {new} minus {base} = effect of the {what}")
    print("=" * 118)
    Q[qid] = {}
    for m, label in (("macro_f1", "macro-F1"), ("class4_recall", "Class 4 recall (diagnostic)")):
        b0 = np.array([float(tab[(s, base)][m]) for s in complete])
        b1 = np.array([float(tab[(s, new)][m]) for s in complete])
        P = paired_stats(b1, b0)
        P["verdict"] = effect_verdict(P)
        Q[qid][m] = P
        print(f"  {label}")
        print(f"    {'seed':>6} {base:>9} {new:>9} {'diff':>9}")
        for s, x0, x1, d in zip(complete, b0, b1, P["d"]):
            print(f"    {s:>6} {x0:>9.4f} {x1:>9.4f} {d:>+9.4f}")
        print(f"    mean paired difference {P['mean']:+.4f} | std of paired differences {P['sd']:.4f} | {new} better on "
              f"{P['wins']}/{n} (worse {P['losses']}, ties {P['ties']})")
        print(f"    paired t-test (PRIMARY) two-sided p = {P['p_t']:.4f} ({'< ' if P['p_t'] < PAIRED_ALPHA else '>= '}"
              f"{PAIRED_ALPHA}) | Wilcoxon signed-rank p = {P['p_w']:.4f} (smallest achievable with n={n}: {2 / 2 ** n:.4f})")
        print(f"    -> {P['verdict']}")

inter = np.array([(float(tab[(s, 'C')]['macro_f1']) - float(tab[(s, 'A')]['macro_f1']))
                  - (float(tab[(s, 'B')]['macro_f1']) - float(tab[(s, 'D')]['macro_f1'])) for s in complete])
print(f"\nINTERACTION (descriptive), macro-F1: (C - A) - (B - D) = {inter.mean():+.4f} ± {inter.std(ddof=1):.4f} across seeds "
      f"-> how much LACA's effect changes when HTT time is present")

# --- H2 retention curves, side by side ---
h2 = pd.read_csv(H2_CSV)
print("\n" + "=" * 118)
print(f"H2 TEMPORAL-SHIFT ROBUSTNESS — macro-F1 retention (%) vs offset, seed {H2_SEED}, tau_s <- clamp(tau_s + delta, 0, 1)")
print("=" * 118)
print(f"  {'condition':<34}" + "".join(f"{'+' + format(o, '.2f'):>10}" for o in H2_OFFSETS) + f"{'min':>9}")
for c, (h, l) in CONDITIONS.items():
    sub = h2[h2["condition"] == c].set_index("offset")
    if sub.empty:
        print(f"  {c:<34} (no H2 data)")
        continue
    ret = [float(sub.loc[o, "retention_pct"]) for o in H2_OFFSETS]
    print(f"  {c + ' (HTT ' + ('ON' if h else 'OFF') + ', LACA ' + ('ON' if l else 'OFF') + ')':<34}"
          + "".join(f"{r:>9.2f}%" for r in ret) + f"{min(ret):>8.2f}%")
print("  (Retention needs timestamps to matter: a condition whose curve stays ~100% is not relying on the absolute "
      "audio<->IMU timing; D must be exactly 100% everywhere.)")

# --- the explicit answer ---
real = lambda qid: Q[qid]["macro_f1"]["verdict"]
q1, q2, q3 = real("Q1"), real("Q2"), real("Q3")
print("\n" + "#" * 118)
print("#  ANSWERS (primary metric macro-F1; Class 4 recall shown as a diagnostic)")
print("#" * 118)
for qid, question, new, base, what in QUESTIONS:
    f, c4 = Q[qid]["macro_f1"], Q[qid]["class4_recall"]
    print(f"#  {qid}. {question}")
    print(f"#      macro-F1: {f['verdict']} — {new}-{base} {f['mean']:+.4f} (std {f['sd']:.4f}, {f['wins']}/{f['n']} seeds, "
          f"p {f['p_t']:.4f}; paired-significant at p<{PAIRED_ALPHA}: {'YES' if f['p_t'] < PAIRED_ALPHA else 'NO'})")
    print(f"#      Class 4 recall: {c4['verdict']} — {c4['mean']:+.4f} (std {c4['sd']:.4f}, {c4['wins']}/{c4['n']}, p {c4['p_t']:.4f})")
if q1 == "REAL POSITIVE" and q2 != "REAL POSITIVE":
    PATTERN = "(ii)"
    msg = ("HTT's simple additive time embedding carries useful temporal information, and LACA's lag bias adds nothing "
           "reliable on top of it (redundant / ineffective)."
           + (" LACA's lag bias does help when HTT is absent (Q3), so it can extract temporal information, but only what "
              "HTT already provides." if q3 == "REAL POSITIVE" else ""))
elif q2 == "REAL POSITIVE" and q3 != "REAL POSITIVE":
    PATTERN = "(iii)"
    msg = "LACA's lag bias is useful, but only together with HTT's time embedding: it adds on top of HTT and does not help alone."
elif q1 != "REAL POSITIVE" and q2 != "REAL POSITIVE" and q3 != "REAL POSITIVE" and "REAL NEGATIVE" not in (q1, q2, q3):
    PATTERN = "(i)"
    msg = ("No temporal path shows a reliable benefit: removing HTT's time embedding does not reliably hurt, and LACA's lag "
           "bias helps neither with nor without it. On this evidence, explicit timestamp information is not shown to be "
           "useful for this task. Strictly this means no benefit was detected, not that none exists.")
else:
    PATTERN = "(iv)"
    parts = [f"Q1 (HTT time) {q1}", f"Q2 (LACA beyond HTT) {q2}", f"Q3 (LACA without HTT) {q3}"]
    msg = "Another pattern: " + "; ".join(parts) + ". "
    if q2 == "REAL POSITIVE" and q3 == "REAL POSITIVE":
        msg += "LACA's lag bias helps both with and without HTT, so it is useful on its own and not redundant with HTT."
    elif "REAL NEGATIVE" in (q1, q2, q3):
        msg += ("At least one temporal component reliably HURTS: " + ", ".join(
            name for name, v in (("HTT time", q1), ("LACA lag beyond HTT", q2), ("LACA lag without HTT", q3))
            if v == "REAL NEGATIVE") + ". Adding that timestamp path makes the model worse on these seeds.")
    else:
        msg += "The pattern does not fit (i)-(iii) cleanly; see the per-question lines above."
print("#")
print(f"#  THE EVIDENCE SUPPORTS PATTERN {PATTERN}: {msg}")
if n < len(SEEDS):
    print(f"#  ⚠️  based on {n}/{len(SEEDS)} complete seed groups only.")
print("#" * 118)

Complete seed groups (all 4 conditions): 5/5 -> [42, 123, 2024, 7, 99]

PER-CONDITION MEANS over seeds [42, 123, 2024, 7, 99]
  condition                           accuracy  macro-F1  bal-acc    AUROC  C3 rec  C4 rec   3->4   4->3
  A (HTT-time ON, LACA-lag OFF)         0.8904    0.8861   0.8828   0.9870  0.8781  0.7801   17.8   41.8
  B (HTT-time OFF, LACA-lag ON)         0.8869    0.8826   0.8795   0.9869  0.8588  0.7970   22.0   38.0
  C (HTT-time ON, LACA-lag ON)          0.8844    0.8796   0.8772   0.9863  0.8368  0.7910   24.0   38.0
  D (HTT-time OFF, LACA-lag OFF)        0.8919    0.8876   0.8845   0.9870  0.8711  0.8080   19.6   35.4

Q1. Does removing HTT's time embedding hurt?  ->  A minus D = effect of the HTT time embedding (LACA off in both)
  macro-F1
      seed         D         A      diff
        42    0.9064    0.9057   -0.0007
       123    0.8948    0.8827   -0.0121
      2024    0.8824    0.8764   -0.0061
         7    0.8835    0.9019   +0.0184
        99    0.87

## 8. Final validation
A recap of every check: data and RNG diagnostics, the Condition A reuse decision, pairing, both switches exactly 0 when OFF, confusion-matrix counts, the H2 curves (D exactly flat), and the pattern.

In [8]:
# ==============================================================================
# FINAL VALIDATION — every verification hook in one place
# ==============================================================================
print("=" * 100)
print("FINAL VALIDATION")
print("=" * 100)
data_checks = {
    "strongest-64 bin indices": SELECTED_BINS.tolist() == REFERENCE_DATA["bin_indices"],
    **{f"{s} windows / sensor values / timestamps / class counts": DATA_FINGERPRINT[s] ==
       REFERENCE_DATA["sensor_fingerprint"][s] for s in ("train", "val", "test")},
    "seed-42 torch RNG after construction == v16/v18": run_results[(42, "A")]["rng_sha1"].startswith(REFERENCE_SEED42_RNG_PREFIX),
    "fusion backbone is real mamba_ssm": MAMBA_CLASS is not None,
}
for name, ok in data_checks.items():
    print(f"  {name:<58}: {'match' if ok else 'MISMATCH'}")
trained = {k: r for k, r in run_results.items() if sources[k] == "trained in v22"}
print("\n" + "=" * 100)
print(f"  Condition A reuse decision          : " + (f"REUSED for {sum(1 for v in sources.values() if v.startswith('reused'))} "
      f"seeds — seed {REUSE_CHECK_SEED} retrained reproduced v18 (max |delta| {REUSE_A_DELTA:.1e})" if REUSE_A else
      f"RETRAINED for every seed (seed {REUSE_CHECK_SEED} max |delta| vs v18 {REUSE_A_DELTA})"))
print(f"  every trained run paired with v18   : {'✅ identical initial weights + RNG (all conditions)' if all(r['paired_with_v18'] for r in trained.values()) else '🚨 ' + str([k for k, r in trained.items() if not r['paired_with_v18']])}")
print(f"  HTT time OFF -> contribution 0      : ✅ asserted before training and at the best checkpoint for "
      f"{sum(1 for (s, c) in trained if not CONDITIONS[c][0])} runs (B, D)")
print(f"  LACA lag OFF -> contribution 0      : ✅ asserted before training, every epoch and at the best checkpoint for "
      f"{sum(1 for (s, c) in trained if not CONDITIONS[c][1])} trained runs (A, D)")
print(f"  LACA lag ON -> active after training: " + ", ".join(f"{c}{s}: {r['lag_term_final']:.3f}" for (s, c), r in trained.items() if CONDITIONS[c][1]))
print(f"  confusion matrices printed          : {len(run_results)} slots ({len(trained)} trained + {len(run_results) - len(trained)} reused)")
print(f"  H2 curves printed                   : {sorted(h2_results)} (seed {H2_SEED}); D exactly flat: "
      + ("✅" if "D" in h2_results and all(r["identical_to_offset0"] for r in h2_results["D"]) else "❌ / missing"))
print(f"  PATTERN                             : {PATTERN}")
print("=" * 100)
if not all(data_checks.values()):
    print("🚨 a data / RNG diagnostic is a MISMATCH — the data or model construction differs from v18.")

FINAL VALIDATION
  strongest-64 bin indices                                  : match
  train windows / sensor values / timestamps / class counts : match
  val windows / sensor values / timestamps / class counts   : match
  test windows / sensor values / timestamps / class counts  : match
  seed-42 torch RNG after construction == v16/v18           : match
  fusion backbone is real mamba_ssm                         : match

  Condition A reuse decision          : REUSED for 4 seeds — seed 42 retrained reproduced v18 (max |delta| 0.0e+00)
  every trained run paired with v18   : ✅ identical initial weights + RNG (all conditions)
  HTT time OFF -> contribution 0      : ✅ asserted before training and at the best checkpoint for 10 runs (B, D)
  LACA lag OFF -> contribution 0      : ✅ asserted before training, every epoch and at the best checkpoint for 6 trained runs (A, D)
  LACA lag ON -> active after training: B42: 1.126, C42: 1.855, B123: 1.299, C123: 1.581, B2024: 0.966, C2024: 0.969, B7: